In [ ]:
%%capture
%run script_nettoyage.ipynb

In [ ]:
# Importation des librairies
import pandas as pd


# Importation des données externes
from data import pop_2010


# Importation des fonctions personnalisées
from fonctions import (
    tracer_comparaison_frequentation,
    beau_tableau,
    generer_carte_musees,
    carto_frequentation_region,
    carto_beaux_arts_themes
)

## 1. Analyse de la fréquentation des musées entre les années

In [ ]:
mes_donnees = {
    "Totale": df_freq_totale,
    "Gratuite": df_freq_gratuite,
    "Payante": df_freq_payante    
}

tracer_comparaison_frequentation(
    mes_donnees, 
    an='annee',           
    freq='freq_net'    
)

La fréquentation totale semble avoir augmenté au fil des ans. La fréquentation de musées de manière payante est plus forte que la fréquentation gratuite.

## 2. Analyse de la fréquentation des musées par région

Dans un premier temps, nous souhaitons étudier le nombres de musées fermés par régions. Fermer un musée pourrait agir comme un frein à la fréquentation.

In [ ]:
# Garde toutes les lignes où le Statut n'est pas la chaîne de caractères 'NA'
df_reg = df_freq_totale[df_freq_totale['Statut'] == 'Fermé'].copy()
df_reg_2014 = df_reg[df_reg['annee'] == '2014']


# Tableau croisé par région
# Remplacer 'region' par le nom exact de votre colonne géographique
tableau_statut_region = pd.crosstab(df_reg_2014['NOMREG'], df_reg_2014['Statut'])


# On trie par ordre croissant
tableau_statut_region_trié = tableau_statut_region.sort_values(by='Fermé', ascending=True)


# Transformation en DataFrame
tableau_statut_region_trié = tableau_statut_region_trié.reset_index()
tableau_statut_region_trié.columns = ['Région', 'Nombre de musées fermés']


# Affichage
beau_tableau(
    tableau_statut_region_trié, "Répartition des musées fermés par régions en 2014"
    )


En 2014, en Nouvelle-Aquitaine, plus de musées étaient fermés.

Ensuite, nous souhaitons analyser le nombre de musées par région. En effet, cela constitue un point d'analyse important : on pourrait penser que plus il y a de musées, plus il y aura de visiteurs. Pour cela, nous allons utiliser la fonction generer_carte_musees, qui permet de générer une carte représentant la répartition du nombre de région pour une année donnée.

In [ ]:
generer_carte_musees(df_freq_totale, '2014')

Nous remarquons une disparités dans le nombre de musées par région : la Bretagne et les région d'Outre-mer en possèdent moins, l'Île-de-France et les régions du sud en possèdent plus. Cela peut avoir une influence sur la fréquentation des musées dans les régions.

Nous souhaitons à présent représenter sur une carte de la France métropolitaine la fréquentation des musées par région. Cependant, nous voulons également représenter la fréquentation par rapport au nombre d'habitants dans la région. Pour cela nous utilisons une table présentant le nombre d'habitants par région en 2010 (fichier data.py). Nous faisons ensuite une jointure de cette table avec notre table fréquentation_totale. Le fait de prendre comme 2010 comme référence pour toutes les années de notre base de données n'est pas précis; l'objectif n'est pas d'être précis mais de montrer les tendances. Nous créons la colonne freq_pour_1000_hab représentant la fréquentation du musée, une année donnée pour 1 000 habitants.

In [ ]:
from data import pop_2010


df_freq_totale = df_freq_totale.merge(pop_2010, on="NOMREG", how="left")
df_freq_totale["freq_pour_1000_hab"] = (
    pd.to_numeric(df_freq_totale["freq_net"], errors="coerce")
    / df_freq_totale["population_2010"] * 1000
)
#df_freq_totale

La fonction carto_frequentation_region permet de représenter sur une carte de la France métropolitaine la fréquentation par région une année donnée par quartiles. L'argument col_freq permet de choisir entre représenter la fréquentation ou la fréquentation pour 1 000 habitants.

In [ ]:
carte_2022 = carto_frequentation_region(df_freq_totale, "2014", col_freq="freq_pour_1000_hab")

Au moins une région dépasse les 1 000 visiteurs pour 1 000 habitants. Qui dépasse ce seuil et en quelles années ?

In [ ]:
freq_region_annee = (
    df_freq_totale
    .groupby(["annee", "NOMREG"], as_index=False)["freq_pour_1000_hab"]
    .sum()
)

freq_region_annee.loc[
    freq_region_annee["freq_pour_1000_hab"] > 1000,
    ["annee", "NOMREG", "freq_pour_1000_hab"]
].sort_values(["NOMREG", "annee"])

La région Ile-De-France dépasse ce seuil tous les ans. Cela s'explique par le tourisme très important que connaît cette région.

Par ailleurs, nous remarquons que le nombre de musées ne semble que peu influencer la fréquentation. En effet, la région Auvergne-Rhône-Alpes présente une fréquentation pour 1000 habitants faible, alors qu'elle abrite un grand nombre de musées (136 musées, ce qui est le deuxième plus haut score). En outre, nous remarquons, que le nombre de musées fermés ne semble pas vraiment influencer la fréquentation.

Ce qui ressort de cette analyse est que l'attractivité touristique génère de plus grandes fréquentations de musées. Les régions comme la Corse, Paris ou encore PACA, montrent une forte fréquentation pour 1000 habitants. La Corse en particulier, va dans ce sens, puisqu'elle n'a que très peu de musées mais répertorie l'un des plus grands nombres de fréquentation pour 1000 habitants.

## 3. Analyse des thématiques qui peuvent influencer la fréquentation de musées

Commençons par étudier le classement des thèmes : du plus abordé au moins abordé.

In [ ]:
# Liste de tes thèmes 
themes = list(colonnes_binaires.columns)


# Liste de colonnes à garder
colonnes_a_garder = ["NOMREG", "IDMuseofile", "annee"] + themes


# On ne garde que certaines colonnes pour faire l'étude
df_musees_themes = df_freq_totale[colonnes_a_garder].copy()


# On crée un DataFrame avec une seule ligne par musée
# On garde l'ID et les thèmes pour le calcul
df_unique_musees = df_musees_themes.drop_duplicates(subset=['IDMuseofile'])


# On sélectionne uniquement les colonnes de thèmes sur ce nouveau DataFrame
df_themes_uniques = df_unique_musees[themes]


# On fait la somme et on trie (cela crée une Series)
classement_them_final = df_themes_uniques.sum().sort_values(ascending=False)


# Transformation en DataFrame et renommage des colonnes pour faire propre
classement_them_final = classement_them_final.reset_index()
classement_them_final.columns = ['Thème', 'Nombre de musées']


# Affichage des 15 premiers thèmes
beau_tableau(
    classement_them_final.head(15), "Classement des 15 premiers thèmes abordés"
    )


La thématique "Beaux-Arts" est très largement supérieure avec 533 musées qui possèdent une collection de Beaux-Arts. L'archéologie se distingue aussi. Les thèmes supérieurs sont : Beaux-arts Archéologie, Arts décoratifs, Technique et industrie, Art moderne et contemporain et, enfin, Sciences de la nature. Les autres thèmes sont beaucoup moins représentés.

À présent, nous aimerions étudier la répartition des musées des Beaux-Arts par région.

In [ ]:
# On garde un seul musée par ligne pour ne pas fausser le compte
df_unique = df_musees_themes.drop_duplicates(subset=['IDMuseofile']).copy()


# On fait la somme du thème par région
stats_region = df_unique.groupby("NOMREG", as_index=False)["Beaux_arts"].sum()


# On trie par la colonne "Beaux_arts" en ordre croissant
stat_classees = stats_region.sort_values(by="Beaux_arts", ascending=False).head(5)
beau_tableau(
    stat_classees, "Classement des 5 premières régions proposant une section Beaux-Arts par région"
    )

In [ ]:
# Pour le thèmes des Beaux-arts
carto_beaux_arts_themes(df_musees_themes, "Beaux_arts")

L'Île-de-France aborde le plus la thématique 'Beaux-arts', mais ce résultat est à relativiser : la région possède aussi plus de musées. La région Hauts-De-France se distingue par son nombre de fois abordées comparé à son nombre de musées : 50 musées qui aborde le thème pour 85 musées.

Nous pouvons maintenant étudier le nombre thèmes abordés par région.

In [ ]:
# On filtre les données pour ne garder que l'année 2014
df_2014 = df_musees_themes[df_musees_themes["annee"] == '2014']


# On garde un seul musée par ligne pour éviter les doublons
df_unique_2014 = df_2014.drop_duplicates(subset=['IDMuseofile'])


# On fait la somme de chaque thème, par région
somme_themes_region = df_unique_2014.groupby("NOMREG")[themes].sum()


# On compte le nombre de thèmes abordés par région
nombre_themes_par_region = (somme_themes_region > 0).sum(axis=1)


# On trie le résultat pour voir la région avec le plus de thèmes en premier
classement_regions = nombre_themes_par_region.sort_values(ascending=False)


# Affichage du résultat
classement_regions


# Transformation en DataFrame
df_regions = classement_regions.reset_index()
df_regions.columns = ['Région', 'Nombre de thèmes']


# Affichage des 10 premiers
beau_tableau(
    df_regions.head(10), "Classement des 10 premières régions selon le nombre de thèmes"
    )

On remarque que le nombre de thèmes ne semble pas corrélé avec la fréquentation de musées ni au nommbre de musées présents dans la région. En effet, une fois encore, la région Auvergne-Rhône-Alpes répertorie le plus grand nombre de thématiques, et pourtant, enregistre une faible fréquentation. De plus, la Bretagne aborde 16 thématiques mais ne possède que très peu de musées.

## Conclusion de partie

Il ressort que le facteur principal qui influence la fréquentation est le tourisme. En effet, nous avons remarqué que les régions montrant une plus haute fréquentation de musées sont aussi les régions touristiques. Nous aurions pu élargir l'analyse au champ du tourisme pour confirmer l'hypothèse.